# Supermarket Sales Analysis Using Python

**Student:** Gulam Mahiuddin  
**Dataset:** `SUPER MARKET DATA(3).xlsx`  
**Purpose:** End-to-end supermarket sales analysis for the AICTE / IBM SkillsBuild internship project.

## Objectives
- Load and validate the supermarket dataset.
- Calculate sales KPIs.
- Analyze city/branch, category, product, monthly, payment, customer and gender performance.
- Validate `Sales = Quantity × Unit Price`.
- Generate business insights with reproducible Python code.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_FILE = Path("SUPER MARKET DATA(3).xlsx")
df = pd.read_excel(DATA_FILE, sheet_name=0)
df.head()


## 1. Data Understanding

In [ ]:
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)
display(df.describe(include="all"))


## 2. Data Quality Checks

In [ ]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Calculated Sales"] = df["Quantity"] * df["Unit Price"]
df["Sales Difference"] = (df["Sales"] - df["Calculated Sales"]).abs()

quality = pd.DataFrame({
    "Check": ["Missing values", "Duplicate rows", "Invalid dates",
              "Maximum Sales calculation difference"],
    "Result": [
        int(df.isna().sum().sum()),
        int(df.duplicated().sum()),
        int(df["Date"].isna().sum()),
        float(df["Sales Difference"].max())
    ]
})
display(quality)


## 3. Feature Engineering

In [ ]:
df["Month"] = df["Date"].dt.to_period("M").astype(str)
df["Average Item Value"] = df["Sales"] / df["Quantity"]
display(df.head())


## 4. KPI Analysis

In [ ]:
kpis = pd.DataFrame({
    "KPI": ["Total Sales", "Transactions", "Total Quantity",
            "Average Transaction Value", "Average Rating"],
    "Value": [
        df["Sales"].sum(),
        df["Invoice ID"].nunique(),
        df["Quantity"].sum(),
        df["Sales"].mean(),
        df["Rating"].mean()
    ]
})
display(kpis)


## 5. City / Branch Performance

In [ ]:
city_summary = df.groupby(["Branch","City"]).agg(
    Sales=("Sales","sum"),
    Quantity=("Quantity","sum"),
    Transactions=("Invoice ID","nunique"),
    Avg_Rating=("Rating","mean")
).sort_values("Sales", ascending=False)
display(city_summary)

city_sales = df.groupby("City")["Sales"].sum().sort_values(ascending=False)
plt.figure(figsize=(9,5))
plt.bar(city_sales.index, city_sales.values)
plt.title("Sales by City")
plt.xlabel("City"); plt.ylabel("Sales")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 6. Category and Product Analysis

In [ ]:
category_summary = df.groupby("Category").agg(
    Sales=("Sales","sum"), Quantity=("Quantity","sum"),
    Transactions=("Invoice ID","nunique")
).sort_values("Sales", ascending=False)

product_summary = df.groupby("Product").agg(
    Sales=("Sales","sum"), Quantity=("Quantity","sum"),
    Transactions=("Invoice ID","nunique")
).sort_values("Sales", ascending=False)

display(category_summary)
display(product_summary.head(10))

plt.figure(figsize=(10,5))
plt.bar(category_summary.index, category_summary["Sales"])
plt.title("Sales by Product Category")
plt.xlabel("Category"); plt.ylabel("Sales")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

top10 = product_summary.head(10).sort_values("Sales")
plt.figure(figsize=(10,6))
plt.barh(top10.index, top10["Sales"])
plt.title("Top 10 Products by Sales")
plt.xlabel("Sales"); plt.ylabel("Product")
plt.tight_layout()
plt.show()


## 7. Monthly Sales Trend

In [ ]:
monthly_summary = df.groupby("Month").agg(
    Sales=("Sales","sum"), Quantity=("Quantity","sum"),
    Transactions=("Invoice ID","nunique")
).sort_index()
display(monthly_summary)

plt.figure(figsize=(10,5))
plt.plot(monthly_summary.index, monthly_summary["Sales"], marker="o")
plt.title("Monthly Sales Trend")
plt.xlabel("Month"); plt.ylabel("Sales")
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()

# July 2026 is partial in this dataset: only 7 transactions are dated July 1.


## 8. Payment Method Analysis

In [ ]:
payment_summary = df.groupby("Payment").agg(
    Sales=("Sales","sum"), Transactions=("Invoice ID","nunique")
).sort_values("Sales", ascending=False)
display(payment_summary)

plt.figure(figsize=(8,5))
plt.bar(payment_summary.index, payment_summary["Sales"])
plt.title("Sales by Payment Method")
plt.xlabel("Payment Method"); plt.ylabel("Sales")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 9. Customer and Gender Analysis

In [ ]:
customer_summary = df.groupby("Customer Type").agg(
    Sales=("Sales","sum"), Transactions=("Invoice ID","nunique"),
    Avg_Rating=("Rating","mean")
).sort_values("Sales", ascending=False)

gender_summary = df.groupby("Gender").agg(
    Sales=("Sales","sum"), Transactions=("Invoice ID","nunique"),
    Avg_Rating=("Rating","mean")
).sort_values("Sales", ascending=False)

display(customer_summary)
display(gender_summary)


## 10. Cross Analysis

In [ ]:
city_category = pd.pivot_table(
    df, index="City", columns="Category", values="Sales",
    aggfunc="sum", fill_value=0
)
display(city_category)

display(df.groupby("City")["Rating"].mean().sort_values(ascending=False).to_frame("Average Rating"))


## 11. Automated Business Insights

In [ ]:
city_sales = df.groupby("City")["Sales"].sum()
category_sales = df.groupby("Category")["Sales"].sum()
product_sales = df.groupby("Product")["Sales"].sum()
monthly_sales = df.groupby("Month")["Sales"].sum()

print(f"Total sales: {df['Sales'].sum():,.2f}")
print(f"Highest-sales city: {city_sales.idxmax()} ({city_sales.max():,.2f})")
print(f"Highest-sales category: {category_sales.idxmax()} ({category_sales.max():,.2f})")
print(f"Highest-sales product: {product_sales.idxmax()} ({product_sales.max():,.2f})")
print(f"Highest recorded month: {monthly_sales.idxmax()} ({monthly_sales.max():,.2f})")
print(f"Average rating: {df['Rating'].mean():.2f}/5")


## 12. Conclusion

The project demonstrates a complete Python data-analytics workflow: ingestion, validation, feature preparation, KPI analysis, aggregation, visualization, cross-analysis and business insight generation.

**Limitation:** July 2026 is only partially represented (7 transactions on July 1), so it should not be treated as a complete month.

**Future AI/ML scope:** demand forecasting, customer segmentation, anomaly detection and natural-language reporting using a larger historical dataset.
